# Bab 7: Evaluasi Model

Notebook pendamping buku *Data Mining dengan Python: Konsep, Algoritme, dan Studi Kasus dengan CRISP-DM*. Setiap sel kode sama dengan listing di buku; keluaran yang dicetak di buku ada di berkas `.out` di folder `kode/bab07/`.

Jalankan sel secara berurutan.

In [ ]:
# Jalankan sel ini sekali di awal sesi.
import os, sys, subprocess
if "google.colab" in sys.modules and not os.path.exists("kode"):
    if not os.path.exists("data-mining-dengan-python"):
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/bsnegara/data-mining-dengan-python"])
    os.chdir("data-mining-dengan-python")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "-r", "kode/requirements-colab.txt"])
while not os.path.exists("kode/data") and os.getcwd() != "/":
    os.chdir("..")                     # cari akar repositori
sys.path.insert(0, "kode/bab07")
print("Folder kerja:", os.getcwd())

## Listing 7.1: Modul bantu bab ini

`kode/bab07/siapkan.py`

In [ ]:
"""Modul bantu Bab 7: memuat data churn dan membuat pipeline standar."""
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

DATA = "kode/data/"
NUMERIK = ["umur", "frekuensi_90h", "frekuensi_90h_lalu", "recency_hari",
           "nilai_90h", "rata_nilai_struk", "porsi_promo", "porsi_aplikasi",
           "porsi_kopi_hitam", "porsi_pagi", "porsi_akhir_pekan",
           "ragam_menu", "keluhan_90h", "jarak_km", "pakai_ulasan"]
KATEGORIK = ["kota", "tingkat", "jenis_kelamin"]


def muat():
    df = pd.read_csv(DATA + "pelanggan.csv")
    return df[NUMERIK + KATEGORIK], df["churn"]


def pipa(model, skala=True):
    num = StandardScaler() if skala else "passthrough"
    pra = ColumnTransformer([
        ("num", num, NUMERIK),
        ("kat", OneHotEncoder(handle_unknown="ignore"), KATEGORIK)])
    return make_pipeline(pra, model)

## Listing 7.2: Akurasi latih, akurasi uji, dan pembanding

`kode/bab07/akurasi_semu.py`

In [ ]:
import sys; sys.modules.pop('siapkan', None); sys.path.insert(0, 'kode/bab07')
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from siapkan import muat, pipa

X, y = muat()
print(f"porsi churn: {y.mean():.4f}")
Xl, Xu, yl, yu = train_test_split(X, y, test_size=0.3,
                                  stratify=y, random_state=7)
model = {
    "tebak mayoritas": DummyClassifier(strategy="most_frequent"),
    "pohon penuh": DecisionTreeClassifier(random_state=0),
    "pohon kedalaman 5": DecisionTreeClassifier(max_depth=5,
                                                random_state=0)}
for nama, m in model.items():
    p = pipa(m, skala=False).fit(Xl, yl)
    print(f"{nama:18s} latih {p.score(Xl, yl):.3f}"
          f"  uji {p.score(Xu, yu):.3f}")

## Listing 7.3: Keragaman hold-out dan validasi silang

`kode/bab07/bagi_data.py`

In [ ]:
import sys; sys.modules.pop('siapkan', None); sys.path.insert(0, 'kode/bab07')
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import (KFold, StratifiedKFold,
                                     cross_val_score, train_test_split)
from siapkan import muat, pipa

X, y = muat()
m = pipa(LogisticRegression(max_iter=2000))

akurasi = []
for benih in range(20):
    Xl, Xu, yl, yu = train_test_split(X, y, test_size=0.3,
                                      random_state=benih)
    akurasi.append(m.fit(Xl, yl).score(Xu, yu))
akurasi = np.array(akurasi)
print(f"hold-out 20 kali : {akurasi.min():.3f} s.d. "
      f"{akurasi.max():.3f} (rata-rata {akurasi.mean():.3f})")

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=0)
skor = cross_val_score(m, X, y, cv=cv)
print(f"CV 10 lipatan    : {skor.mean():.3f} +/- {skor.std():.3f}")

for nama, pemecah in [("KFold", KFold(10, shuffle=True,
                                      random_state=3)),
                      ("StratifiedKFold", cv)]:
    porsi = [y.iloc[iu].mean() for _, iu in pemecah.split(X, y)]
    print(f"{nama:16s} porsi churn per lipatan: "
          f"{min(porsi):.3f} s.d. {max(porsi):.3f}")

## Listing 7.4: Kurva validasi kedalaman pohon

`kode/bab07/bias_varians.py`

In [ ]:
import sys; sys.modules.pop('siapkan', None); sys.path.insert(0, 'kode/bab07')
import pandas as pd
from sklearn.model_selection import StratifiedKFold, validation_curve
from sklearn.tree import DecisionTreeClassifier
from siapkan import muat, pipa

X, y = muat()
cv = StratifiedKFold(5, shuffle=True, random_state=0)
kedalaman = list(range(1, 16))
latih, valid = validation_curve(
    pipa(DecisionTreeClassifier(random_state=0), skala=False), X, y,
    param_name="decisiontreeclassifier__max_depth",
    param_range=kedalaman, cv=cv, scoring="roc_auc")
tab = pd.DataFrame({"kedalaman": kedalaman,
                    "latih": latih.mean(1), "valid": valid.mean(1),
                    "sd": valid.std(1)})
print(tab.round(3).to_string(index=False))
terbaik = tab.loc[tab["valid"].idxmax()]
print(f"terbaik: kedalaman {int(terbaik['kedalaman'])}, "
      f"AUC valid {terbaik['valid']:.3f}")
tab.to_csv("gambar/bab07/kedalaman.dat", sep=" ", index=False)

## Listing 7.5: Metrik klasifikasi dan pengaruh ambang

`kode/bab07/metrik.py`

In [ ]:
import sys; sys.modules.pop('siapkan', None); sys.path.insert(0, 'kode/bab07')
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (balanced_accuracy_score, confusion_matrix,
                             f1_score, matthews_corrcoef,
                             precision_score, recall_score)
from sklearn.model_selection import train_test_split
from siapkan import muat, pipa

X, y = muat()
Xl, Xu, yl, yu = train_test_split(X, y, test_size=0.3, stratify=y,
                                  random_state=7)
m = pipa(LogisticRegression(max_iter=2000)).fit(Xl, yl)
p = m.predict_proba(Xu)[:, 1]
yp = (p >= 0.5).astype(int)
tn, fp, fn, tp = confusion_matrix(yu, yp).ravel()
print(f"TP={tp} FP={fp} FN={fn} TN={tn}")
print(f"akurasi  {(tp + tn) / len(yu):.3f}")
print(f"presisi  {precision_score(yu, yp):.3f}")
print(f"recall   {recall_score(yu, yp):.3f}")
print(f"spesifik {tn / (tn + fp):.3f}")
print(f"F1       {f1_score(yu, yp):.3f}")
print(f"bal.akur {balanced_accuracy_score(yu, yp):.3f}")
print(f"MCC      {matthews_corrcoef(yu, yp):.3f}")

print("\nambang  presisi  recall  F1     ditandai")
for t in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7]:
    yp = (p >= t).astype(int)
    print(f" {t:.1f}    {precision_score(yu, yp):.3f}    "
          f"{recall_score(yu, yp):.3f}   {f1_score(yu, yp):.3f}  "
          f"{yp.sum():5d}")

## Listing 7.6: Kurva ROC dan presisi–recall untuk tiga model

`kode/bab07/roc_pr.py`

In [ ]:
import sys; sys.modules.pop('siapkan', None); sys.path.insert(0, 'kode/bab07')
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, precision_recall_curve,
                             roc_auc_score, roc_curve)
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from siapkan import muat, pipa

X, y = muat()
Xl, Xu, yl, yu = train_test_split(X, y, test_size=0.3, stratify=y,
                                  random_state=7)
model = {"logistik": pipa(LogisticRegression(max_iter=2000)),
         "pohon": pipa(DecisionTreeClassifier(max_depth=5,
                                              random_state=0), False),
         "hutan": pipa(RandomForestClassifier(300, min_samples_leaf=5,
                       random_state=0, n_jobs=-1), False)}
grid = np.linspace(0, 1, 101)
roc, pr = {"fpr": grid}, {"recall": grid}
for nama, m in model.items():
    p = m.fit(Xl, yl).predict_proba(Xu)[:, 1]
    fpr, tpr, _ = roc_curve(yu, p)
    pre, rec, _ = precision_recall_curve(yu, p)
    print(f"{nama:9s} AUC {roc_auc_score(yu, p):.3f}  "
          f"AP {average_precision_score(yu, p):.3f}")
    roc[nama] = np.interp(grid, fpr, tpr)
    urut = np.argsort(rec)
    pr[nama] = np.interp(grid, rec[urut], np.maximum.accumulate(
        pre[urut][::-1])[::-1])
print(f"garis dasar PR = porsi churn = {yu.mean():.3f}")
pd.DataFrame(roc).round(4).to_csv("gambar/bab07/roc.dat", sep=" ",
                                  index=False)
pd.DataFrame(pr).round(4).to_csv("gambar/bab07/pr.dat", sep=" ",
                                 index=False)

## Listing 7.7: Metrik regresi untuk belanja bulan depan

`kode/bab07/metrik_regresi.py`

In [ ]:
import sys; sys.modules.pop('siapkan', None); sys.path.insert(0, 'kode/bab07')
import numpy as np
import pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (mean_absolute_error,
                             mean_absolute_percentage_error,
                             mean_squared_error, r2_score)
from sklearn.model_selection import train_test_split
from siapkan import DATA, muat, pipa

X, _ = muat()
y = pd.read_csv(DATA + "pelanggan.csv")["belanja_bulan_depan"] / 1000
print(f"target (ribu Rp): rata-rata {y.mean():.1f}, median "
      f"{y.median():.1f}, nol {(y == 0).mean():.1%}")
Xl, Xu, yl, yu = train_test_split(X, y, test_size=0.3, random_state=7)
model = {"rata-rata": DummyRegressor(),
         "linier": LinearRegression(),
         "hutan": RandomForestRegressor(300, min_samples_leaf=5,
                                        random_state=0, n_jobs=-1)}
print("model        MAE    RMSE    R2    MAPE*")
for nama, m in model.items():
    yp = pipa(m).fit(Xl, yl).predict(Xu)
    pos = yu > 0
    print(f"{nama:10s} {mean_absolute_error(yu, yp):6.1f} "
          f"{np.sqrt(mean_squared_error(yu, yp)):7.1f} "
          f"{r2_score(yu, yp):5.3f} "
          f"{mean_absolute_percentage_error(yu[pos], yp[pos]):6.1%}")

## Listing 7.8: Pencarian kisi, pencarian acak, dan validasi silang bersarang

`kode/bab07/setel.py`

In [ ]:
import sys; sys.modules.pop('siapkan', None); sys.path.insert(0, 'kode/bab07')
import time
from scipy.stats import randint
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import (GridSearchCV, RandomizedSearchCV,
                                     StratifiedKFold, cross_val_score)
from siapkan import muat, pipa

X, y = muat()
cv = StratifiedKFold(5, shuffle=True, random_state=0)
hutan = pipa(RandomForestClassifier(200, random_state=0, n_jobs=-1),
             skala=False)
awalan = "randomforestclassifier__"
kisi = {awalan + "max_features": [0.2, 0.4, 0.6],
        awalan + "min_samples_leaf": [1, 5, 20, 50]}
t = time.time()
g = GridSearchCV(hutan, kisi, cv=cv, scoring="roc_auc").fit(X, y)
print(f"grid  : {len(g.cv_results_['params'])} kombinasi, "
      f"AUC {g.best_score_:.4f}, {time.time() - t:.0f} dtk")
print("        terbaik:", {k.replace(awalan, ""): v
                           for k, v in g.best_params_.items()})

sebaran = {awalan + "max_features": [0.1, 0.2, 0.3, 0.4, 0.6, 0.8],
           awalan + "min_samples_leaf": randint(1, 80)}
t = time.time()
r = RandomizedSearchCV(hutan, sebaran, n_iter=12, cv=cv,
                       scoring="roc_auc", random_state=0).fit(X, y)
print(f"acak  : 12 kombinasi, AUC {r.best_score_:.4f}, "
      f"{time.time() - t:.0f} dtk")
print("        terbaik:", {k.replace(awalan, ""): v
                           for k, v in r.best_params_.items()})

dalam = GridSearchCV(hutan, kisi, cv=3, scoring="roc_auc")
luar = cross_val_score(dalam, X, y, cv=cv, scoring="roc_auc")
print(f"nested CV: AUC {luar.mean():.4f} +/- {luar.std():.4f}")

## Listing 7.9: Lift, gain, dan kurva laba program voucer

`kode/bab07/laba.py`

In [ ]:
import sys; sys.modules.pop('siapkan', None); sys.path.insert(0, 'kode/bab07')
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from siapkan import muat, pipa

VOUCER, SELAMAT, LABA = 15_000, 0.30, 180_000
ANGGOTA = 26_000
X, y = muat()
Xl, Xu, yl, yu = train_test_split(X, y, test_size=0.3, stratify=y,
                                  random_state=7)
p = pipa(LogisticRegression(max_iter=2000)).fit(Xl, yl).predict_proba(
    Xu)[:, 1]
urut = np.argsort(-p)
churn = yu.to_numpy()[urut]
n = len(churn)
k = np.arange(1, n + 1)
presisi = churn.cumsum() / k
tangkap = churn.cumsum() / churn.sum()
lift = presisi / churn.mean()
manfaat = SELAMAT * LABA * churn.cumsum() - VOUCER * k
laba = manfaat / n * ANGGOTA / 1e6
porsi = k / n
print(f"titik impas presisi = {VOUCER / (SELAMAT * LABA):.3f}")
for q in [0.02, 0.0385, 0.05, 0.10, 0.20, 0.50, 1.0]:
    i = int(round(q * n)) - 1
    print(f"top {q:5.1%}: presisi {presisi[i]:.3f} lift {lift[i]:4.2f}"
          f" tangkap {tangkap[i]:.3f} Rp{laba[i]:6.1f} jt")
i = int(np.argmax(laba))
print(f"laba maksimum Rp{laba[i]:.1f} jt pada top {porsi[i]:.1%} "
      f"(~{porsi[i] * ANGGOTA:.0f} anggota)")
print(f"  presisi {presisi[i]:.3f}, skor ambang {p[urut][i]:.3f}")
acak = SELAMAT * LABA * yu.mean() - VOUCER
print(f"voucer acak: Rp{acak:,.0f} per anggota")
pilih = np.unique(np.r_[np.linspace(0, n - 1, 120).astype(int)])
pd.DataFrame({"porsi": porsi[pilih], "tangkap": tangkap[pilih],
              "lift": lift[pilih], "laba": laba[pilih]}).round(4).to_csv(
    "gambar/bab07/laba.dat", sep=" ", index=False)

## Listing 7.10: Uji 5$times$2cv dan McNemar

`kode/bab07/banding.py`

In [ ]:
import sys; sys.modules.pop('siapkan', None); sys.path.insert(0, 'kode/bab07')
import numpy as np
from scipy import stats
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from siapkan import muat, pipa

X, y = muat()
a = pipa(LogisticRegression(max_iter=2000))
b = pipa(RandomForestClassifier(300, min_samples_leaf=5,
                                random_state=0, n_jobs=-1), False)

d = []
for r in range(5):
    cv = StratifiedKFold(2, shuffle=True, random_state=r)
    beda = []
    for il, iu in cv.split(X, y):
        sa = roc_auc_score(y.iloc[iu], clone(a).fit(X.iloc[il],
                           y.iloc[il]).predict_proba(X.iloc[iu])[:, 1])
        sb = roc_auc_score(y.iloc[iu], clone(b).fit(X.iloc[il],
                           y.iloc[il]).predict_proba(X.iloc[iu])[:, 1])
        beda.append(sa - sb)
    d.append(beda)
d = np.array(d)
s2 = ((d - d.mean(1, keepdims=True)) ** 2).sum(1)
t = d[0, 0] / np.sqrt(s2.mean())
p = 2 * stats.t.sf(abs(t), df=5)
print(f"selisih AUC rata-rata (log - hutan): {d.mean():+.4f}")
print(f"uji 5x2cv: t = {t:.3f}, p = {p:.3f}")

Xl, Xu, yl, yu = train_test_split(X, y, test_size=0.3, stratify=y,
                                  random_state=7)
ba = a.fit(Xl, yl).predict(Xu) == yu
bb = b.fit(Xl, yl).predict(Xu) == yu
n01, n10 = int((ba & ~bb).sum()), int((~ba & bb).sum())
chi = (abs(n01 - n10) - 1) ** 2 / (n01 + n10)
print(f"McNemar: hanya log benar {n01}, hanya hutan benar {n10}")
print(f"         chi2 = {chi:.3f}, p = {stats.chi2.sf(chi, 1):.3f}")

## Listing 7.11: Studi kasus evaluasi model churn

`kode/bab07/studi_kasus.py`

In [ ]:
import sys; sys.modules.pop('siapkan', None); sys.path.insert(0, 'kode/bab07')
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import (StratifiedKFold, cross_val_score,
                                     train_test_split)
from sklearn.tree import DecisionTreeClassifier
from siapkan import muat, pipa


X, y = muat()
Xl, Xu, yl, yu = train_test_split(X, y, test_size=0.3, stratify=y,
                                  random_state=7)
print("model            AUC CV (10 lipatan)")
print(f"aturan recency   {roc_auc_score(yl, Xl['recency_hari']):.3f}"
      "  (tanpa pelatihan)")
kandidat = {
    "pohon (d=5)": pipa(DecisionTreeClassifier(max_depth=5,
                                               random_state=0), False),
    "hutan acak": pipa(RandomForestClassifier(300, max_features=0.2,
                       min_samples_leaf=5, random_state=0, n_jobs=-1),
                       False),
    "logistik": pipa(LogisticRegression(max_iter=2000))}
cv = StratifiedKFold(10, shuffle=True, random_state=0)
for nama, m in kandidat.items():
    s = cross_val_score(m, Xl, yl, cv=cv, scoring="roc_auc")
    print(f"{nama:16s} {s.mean():.3f} +/- {s.std():.3f}")


def presisi_atas(y, p, q=0.0385):
    k = int(round(q * len(y)))
    return y[np.argsort(-p)[:k]].mean()


print("\ndata uji (dikunci sampai sekarang):")
yu_ = yu.to_numpy()
skor = {"aturan recency": Xu["recency_hari"].to_numpy() / 90,
        "logistik": kandidat["logistik"].fit(Xl, yl).predict_proba(
            Xu)[:, 1]}
for nama, p in skor.items():
    print(f"{nama:16s} AUC {roc_auc_score(yu_, p):.3f}  presisi top "
          f"3,85% {presisi_atas(yu_, p):.3f}")

rng = np.random.default_rng(0)
auc, pre = [], []
for _ in range(2000):
    i = rng.integers(0, len(yu_), len(yu_))
    auc.append(roc_auc_score(yu_[i], p[i]))
    pre.append(presisi_atas(yu_[i], p[i]))
print("selang 95% logistik: AUC "
      f"[{np.percentile(auc, 2.5):.3f}, {np.percentile(auc, 97.5):.3f}]"
      f", presisi [{np.percentile(pre, 2.5):.3f}, "
      f"{np.percentile(pre, 97.5):.3f}]")